# Lab 18-10 · Noise Floors, Sensor Fusion and a Synthetic PPG

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sensors as sn

## Part 1 · Noise floor

*Predict* the floor when the proof mass doubles, and when Q rises from 2 to 200.

In [2]:
for m, q in ((2e-9, 2.0), (4e-9, 2.0), (2e-9, 200.0)):
    print(f"m = {m * 1e9:.0f} ug, Q = {q:5.0f}: {sn.brownian_noise_density(m, 4e3, q) / sn.G0 * 1e6:6.2f} ug/rtHz")

m = 2 ug, Q =     2:  32.90 ug/rtHz
m = 4 ug, Q =     2:  23.26 ug/rtHz
m = 2 ug, Q =   200:   3.29 ug/rtHz


## Part 2 · Fusion

*Predict* the fused error for alpha = 0.9, 0.98 and 0.998, and where the minimum lies.

In [3]:
for a in (0.9, 0.98, 0.998):
    acc, gyro, fused = sn.simulate_fusion(a)
    print(f"alpha {a}: tau {sn.complementary_time_constant(a, 0.01):.2f} s, fused rms {fused:.3f} deg")
grid = np.linspace(0.90, 0.995, 40)
errs = [sn.simulate_fusion(a)[2] for a in grid]
plt.plot(grid, errs, "o-"); plt.xlabel("alpha"); plt.ylabel("fused rms error (deg)"); plt.show()
print("best alpha", round(float(grid[int(np.argmin(errs))]), 3), "error", round(min(errs), 3))

alpha 0.9: tau 0.09 s, fused rms 0.263 deg
alpha 0.98: tau 0.49 s, fused rms 0.288 deg
alpha 0.998: tau 4.99 s, fused rms 2.512 deg
best alpha 0.956 error 0.208


## Part 3 · Heart rate

*Predict* the artefact size at which the estimate locks onto the 1.5 Hz motion.

In [4]:
fs, t = 50, np.arange(0, 8, 1 / 50)
rng = np.random.default_rng(1810)
for art in (0.0, 0.5, 0.9, 1.1, 2.0):
    x = (1e-6 * (1 + 0.01 * np.sin(2 * np.pi * 1.2 * t)) + art * 0.01e-6 * np.sin(2 * np.pi * 1.5 * t)
         + rng.normal(0, math.sqrt(2 * sn.Q_E * 1e-6 * 25), len(t)))
    print(f"artefact {art:.1f} x pulse: {sn.heart_rate_from_signal(x, fs):.1f} bpm")

artefact 0.0 x pulse: 71.8 bpm
artefact 0.5 x pulse: 71.8 bpm
artefact 0.9 x pulse: 71.8 bpm
artefact 1.1 x pulse: 90.1 bpm
artefact 2.0 x pulse: 90.1 bpm


## Part 4 · Altitude

*Predict* how clearly a 3 m climb shows in 1 s averages of a 1.5 Pa-noise barometer.

In [5]:
fs, t = 25, np.arange(0, 60, 1 / 25)
rng = np.random.default_rng(1811)
h_true = np.clip((t - 20) / 20, 0, 1) * 3.0
p = 101325.0 + sn.pressure_gradient_pa_per_m() * h_true + rng.normal(0, 1.5, len(t))
p1 = p.reshape(-1, 25).mean(axis=1)
alt = [sn.altitude_m(x) for x in p1]
plt.plot(np.arange(60), alt, "o-"); plt.xlabel("time (s)"); plt.ylabel("altitude (m)"); plt.show()
print("noise of 1 s averages before the climb:", round(float(np.std(alt[:20])) * 100, 1), "cm; step:", round(float(np.mean(alt[45:]) - np.mean(alt[:20])), 2), "m")

noise of 1 s averages before the climb: 2.5 cm; step: 3.02 m


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).